# Diffusion Inpainting Augmentation for Medical Image Classification
## **Key Notebook**

This Key notebook mirrors the student assignment workflow.

It uses the provided checkpoint files as **pretrained starting points**, then performs a short fine-tuning run to demonstrate the complete workflow:

1. Baseline CNN on the artificially imbalanced training set
2. Traditional augmentation CNN with realistic, label-preserving image augmentation
3. Diffusion inpainting augmentation CNN using generated minority-class samples

Validation and test sets are never modified. Diffusion-generated samples are added only to the training set.

The metrics printed by this Key notebook are computed from this notebook run after fine-tuning. The checkpoint files are only used as starting models.


## Background

**PneumoniaMNIST**: binary classification, 64×64 grayscale chest X-rays. Classes: normal (0), pneumonia (1).

**Data Rules:** imbalance → training only; generated samples → training only; augmentation → training time only; val/test → never modified. All metrics are computed on the unchanged test set.

## Setup

In [ ]:
import os, random
import numpy as np
import tensorflow as tf
import keras
from keras import layers
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## Global Configuration

In [ ]:
# ── Reproducibility ──
SEED = 42

def reset_all_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        keras.utils.set_random_seed(seed)
    except Exception:
        pass

reset_all_seeds(SEED)

# ── Core constants ──
IMG_SIZE = 64
BATCH_SIZE = 32
KEY_FINE_TUNE_EPOCHS = 5
KEY_CLASSIFIER_LR = 5e-7
KEY_DENOISER_LR = 2e-6

MINORITY_CLASS = 1                 # Pneumonia, verified against MedMNIST INFO below
MINORITY_KEEP_FRACTION = 0.20      # Artificially keep 20% of pneumonia samples in train
TARGET_MINORITY_MULTIPLIER = 2     # Generate enough samples to double current minority count

DIFFUSION_TIMESTEPS = 50
NOISE_BETA_START = 1e-4
NOISE_BETA_END = 0.02
INPAINT_MASK_FRAC = 0.25

# ── Final gentle traditional augmentation config ──
TRAD_ROTATION_FACTOR = 0.025       # about ±9 degrees
TRAD_TRANSLATION_FACTOR = 0.040    # about ±4% shift
TRAD_ZOOM_FACTOR = 0.050           # about ±5% zoom
TRAD_CONTRAST_FACTOR = 0.10        # mild contrast variation
TRAD_FILL_MODE = "nearest"         # avoids artificial black borders

# ── Provided checkpoint paths used by this Key notebook ──
CHECKPOINT_DIR = "checkpoints"
DENOISER_PRETRAINED_PATH = os.path.join(CHECKPOINT_DIR, "denoiser_final.keras")
BASELINE_PRETRAINED_PATH = os.path.join(CHECKPOINT_DIR, "baseline_cnn_final.keras")
TRAD_AUG_PRETRAINED_PATH = os.path.join(CHECKPOINT_DIR, "traditional_aug_cnn_final.keras")
DIFFUSION_AUG_PRETRAINED_PATH = os.path.join(CHECKPOINT_DIR, "diffusion_aug_cnn_final.keras")


print("Checkpoint directory:", os.path.abspath(CHECKPOINT_DIR))
print("Fine-tuning epochs in Key:", KEY_FINE_TUNE_EPOCHS)


In [ ]:
required_checkpoints = [
    DENOISER_PRETRAINED_PATH,
    BASELINE_PRETRAINED_PATH,
    TRAD_AUG_PRETRAINED_PATH,
    DIFFUSION_AUG_PRETRAINED_PATH,
]

missing = [p for p in required_checkpoints if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Missing required checkpoint(s):\n" + "\n".join(f"  - {p}" for p in missing) +
        "\n\nCreate or upload the complete checkpoints/ folder before running this Key notebook. "
        "The Key notebook uses only model checkpoints and computes its own metrics from this run."
    )

print("✓ All required model checkpoints found.")
for p in required_checkpoints:
    print(f"  - {p}")


## Provided Helper Functions

In [ ]:
# ── Plotting helpers ──
def plot_class_distribution(labels, title="Class Distribution"):
    counts = np.bincount(labels.astype(np.int32), minlength=2)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(["normal (0)", "pneumonia (1)"], counts)
    for i, v in enumerate(counts):
        ax.text(i, v + max(5, counts.max() * 0.01), str(v), ha="center", fontweight="bold")
    ax.set_title(title)
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()


def plot_image_grid(images, n=8, title="Images"):
    n = min(n, len(images))
    cols = min(n, 8)
    rows = max(1, (n + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.5))
    axes = np.array(axes).reshape(-1)
    for i in range(n):
        img = images[i].numpy().squeeze() if hasattr(images[i], "numpy") else np.squeeze(images[i])
        axes[i].imshow(img, cmap="gray")
        axes[i].axis("off")
    for i in range(n, len(axes)):
        axes[i].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(cm, model_name="Model"):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["normal", "pneumonia"])
    ax.set_yticklabels(["normal", "pneumonia"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(model_name)
    threshold = cm.max() / 2 if cm.size and cm.max() > 0 else 0
    for i in range(2):
        for j in range(2):
            ax.text(
                j, i, str(cm[i][j]), ha="center", va="center",
                color="white" if cm[i][j] > threshold else "black", fontsize=14
            )
    plt.tight_layout()
    plt.show()


def plot_original_samples(images, labels, n=8, title="Original Samples"):
    label_names = {0: "normal", 1: "pneumonia"}
    n = min(n, len(images))
    cols = min(n, 8)
    rows = max(1, (n + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.8, rows * 2))
    axes = np.array(axes).reshape(-1)
    for i in range(n):
        axes[i].imshow(np.squeeze(images[i]), cmap="gray")
        axes[i].axis("off")
        axes[i].set_title(label_names.get(int(labels[i]), str(labels[i])), fontsize=9)
    for i in range(n, len(axes)):
        axes[i].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_traditional_aug_examples(images, aug_pipeline, n=4, repeats=3):
    n = min(n, len(images))
    fig, axes = plt.subplots(n, repeats + 1, figsize=((repeats + 1) * 1.8, n * 1.8))
    if n == 1:
        axes = axes[np.newaxis, :]
    for i in range(n):
        axes[i, 0].imshow(np.squeeze(images[i]), cmap="gray")
        axes[i, 0].axis("off")
        if i == 0:
            axes[i, 0].set_title("Original", fontsize=9)
        for j in range(repeats):
            aug = aug_pipeline(images[i:i + 1], training=True)[0]
            axes[i, j + 1].imshow(np.squeeze(aug.numpy()), cmap="gray")
            axes[i, j + 1].axis("off")
            if i == 0:
                axes[i, j + 1].set_title(f"Aug {j + 1}", fontsize=9)
    fig.suptitle("Traditional Augmentation Examples (training-time only)")
    plt.tight_layout()
    plt.show()


def plot_prediction_examples(model, images, labels, n=8, title="Prediction Examples"):
    label_names = {0: "normal", 1: "pneumonia"}
    n = min(n, len(images))
    preds_prob = model.predict(images[:n].astype(np.float32), verbose=0).squeeze()
    preds = (preds_prob >= 0.5).astype(int)
    cols = min(n, 8)
    rows = max(1, (n + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2.5))
    axes = np.array(axes).reshape(-1)
    for i in range(n):
        axes[i].imshow(np.squeeze(images[i]), cmap="gray")
        axes[i].axis("off")
        true_lbl = label_names.get(int(labels[i]), str(labels[i]))
        pred_lbl = label_names.get(int(preds[i]), str(preds[i]))
        color = "green" if preds[i] == int(labels[i]) else "red"
        axes[i].set_title(f"T:{true_lbl}\nP:{pred_lbl} ({preds_prob[i]:.2f})", fontsize=8, color=color)
    for i in range(n, len(axes)):
        axes[i].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


# ── tf.data ──
def make_tf_dataset(images, labels, batch_size=BATCH_SIZE, shuffle=True, augment_fn=None):
    images = images.astype(np.float32)
    labels = labels.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        ds = ds.shuffle(len(images), seed=SEED, reshuffle_each_iteration=True)
    if augment_fn is not None:
        ds = ds.map(lambda x, y: (augment_fn(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


# ── Evaluation ──
def evaluate_model(model, images, labels, model_name="Model"):
    preds_prob = model.predict(images.astype(np.float32), verbose=0).squeeze()
    preds = (preds_prob >= 0.5).astype(np.int32)
    labels = labels.astype(np.int32)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    prec = precision_score(labels, preds, pos_label=MINORITY_CLASS, zero_division=0)
    rec = recall_score(labels, preds, pos_label=MINORITY_CLASS, zero_division=0)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    print(f"\n{'=' * 50}\n  {model_name} — Test Evaluation\n{'=' * 50}")
    print(f"  Accuracy: {acc:.4f}  Macro-F1: {f1:.4f}")
    print(f"  Minority Precision: {prec:.4f}  Minority Recall: {rec:.4f}")
    print(classification_report(labels, preds, target_names=["normal", "pneumonia"], zero_division=0))
    return {
        "accuracy": float(acc),
        "macro_f1": float(f1),
        "minority_precision": float(prec),
        "minority_recall": float(rec),
        "confusion_matrix": cm.tolist(),
    }


# ── Checkpoint validation (light; does not change saved checkpoints) ──
def validate_checkpoint(path, is_denoiser=False):
    assert os.path.exists(path), f"Missing checkpoint: {path}"
    tmp = keras.saving.load_model(path)
    print(f"  ✓ Loaded {path}")
    if is_denoiser:
        d_img = np.random.rand(2, IMG_SIZE, IMG_SIZE, 1).astype(np.float32)
        d_t = np.array([0, 1], dtype=np.int32)
        pred = tmp.predict([d_img, d_t], verbose=0)
        assert pred.shape == (2, IMG_SIZE, IMG_SIZE, 1)
        assert np.all(np.isfinite(pred))
    else:
        d = np.random.rand(2, IMG_SIZE, IMG_SIZE, 1).astype(np.float32)
        pred = tmp.predict(d, verbose=0)
        assert pred.shape == (2, 1)
        assert np.all(np.isfinite(pred))
        assert np.all((pred >= 0) & (pred <= 1))
    print(f"  ✓ Validated {path}")
    del tmp


def compile_classifier_for_key_finetune(model):
    for layer in model.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(KEY_CLASSIFIER_LR),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_augmentation_pipeline():
    return keras.Sequential([
        layers.RandomRotation(TRAD_ROTATION_FACTOR, fill_mode=TRAD_FILL_MODE, seed=SEED),
        layers.RandomTranslation(
            TRAD_TRANSLATION_FACTOR,
            TRAD_TRANSLATION_FACTOR,
            fill_mode=TRAD_FILL_MODE,
            seed=SEED,
        ),
        layers.RandomZoom(TRAD_ZOOM_FACTOR, fill_mode=TRAD_FILL_MODE, seed=SEED),
        layers.RandomContrast(TRAD_CONTRAST_FACTOR, seed=SEED),
    ], name="traditional_augmentation")

def plot_augmentation_strategy_comparison(source_images, aug_pipeline, generated_images, n=6):
    """Compare original, traditional augmentation, and diffusion-generated minority examples.

    The traditional column is an augmented version of the original image in the same row.
    The diffusion column is sampled from the generated minority-image pool, so it shows the
    diffusion augmentation distribution rather than a strict one-to-one transform of the same row.
    """
    n = min(n, len(source_images), len(generated_images))
    if n == 0:
        print("No images available for augmentation strategy comparison.")
        return

    fig, axes = plt.subplots(n, 3, figsize=(7.5, 2.2 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i in range(n):
        original = np.squeeze(source_images[i])
        traditional = aug_pipeline(source_images[i:i + 1], training=True)[0].numpy()
        diffusion = generated_images[i]

        axes[i, 0].imshow(original, cmap="gray")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(np.squeeze(np.clip(traditional, 0.0, 1.0)), cmap="gray")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(np.squeeze(np.clip(diffusion, 0.0, 1.0)), cmap="gray")
        axes[i, 2].axis("off")

    axes[0, 0].set_title("Original Minority", fontsize=10)
    axes[0, 1].set_title("Traditional Aug", fontsize=10)
    axes[0, 2].set_title("Diffusion Generated", fontsize=10)

    fig.suptitle(
        "Augmentation Strategy Comparison\n"
        "Traditional = same source image transformed; Diffusion = generated minority samples",
        fontsize=12
    )
    plt.tight_layout()
    plt.show()



## Part 1 — Load PneumoniaMNIST and Create Artificial Imbalance [15 pts]

In [ ]:
# ==========================================================
# Solution 1.1: Load PneumoniaMNIST dataset [5 pts]
# ==========================================================
from medmnist import PneumoniaMNIST, INFO

info = INFO["pneumoniamnist"]
label_map = info["label"]
print("Label mapping:", label_map)
assert label_map[str(MINORITY_CLASS)] == "pneumonia", "MINORITY_CLASS mismatch!"

splits = {}
for split_name in ["train", "val", "test"]:
    try:
        ds = PneumoniaMNIST(split=split_name, download=True, size=IMG_SIZE)
        images = ds.imgs.astype(np.float32) / 255.0
        if images.ndim == 3: images = images[..., np.newaxis]
        print(f"  {split_name}: loaded at size={IMG_SIZE}")
    except Exception:
        ds = PneumoniaMNIST(split=split_name, download=True)
        images = ds.imgs.astype(np.float32) / 255.0
        if images.ndim == 3: images = images[..., np.newaxis]
        images = tf.image.resize(images, [IMG_SIZE, IMG_SIZE]).numpy()
        print(f"  {split_name}: resized to {IMG_SIZE}×{IMG_SIZE}")
    labels = ds.labels.squeeze().astype(np.int32)
    splits[split_name] = (images, labels)
    print(f"    shape={images.shape}, counts={np.bincount(labels)}")

train_images, train_labels = splits["train"]
val_images, val_labels = splits["val"]
test_images, test_labels = splits["test"]

In [ ]:
# ==========================================================
# Solution 1.2: Create artificial imbalance [5 pts]
# ==========================================================
def create_imbalance(images, labels, minority_class=MINORITY_CLASS,
                     keep_fraction=MINORITY_KEEP_FRACTION, seed=SEED):
    rng = np.random.RandomState(seed)
    mi = labels == minority_class; ma = ~mi
    mi_imgs, mi_lbls = images[mi], labels[mi]
    n_keep = int(len(mi_imgs) * keep_fraction)
    idx = rng.choice(len(mi_imgs), n_keep, replace=False)
    out_imgs = np.concatenate([images[ma], mi_imgs[idx]])
    out_lbls = np.concatenate([labels[ma], mi_lbls[idx]])
    perm = rng.permutation(len(out_imgs))
    return out_imgs[perm], out_lbls[perm]

imb_train_images, imb_train_labels = create_imbalance(train_images, train_labels)
print(f"Before: {np.bincount(train_labels)}")
print(f"After:  {np.bincount(imb_train_labels)}")

In [ ]:
# ==========================================================
# Solution 1.3: Visualize original samples and distributions [5 pts]
# ==========================================================
plot_original_samples(train_images, train_labels, n=8, title="Original Training Samples")
plot_class_distribution(train_labels, "Training Set — Before Imbalance")
plot_class_distribution(imb_train_labels, "Training Set — After Imbalance")

## Part 2 — Baseline CNN with Checkpoint Loading and Fine-Tuning [20 pts]

The baseline model starts from the provided baseline checkpoint and is fine-tuned briefly on the artificially imbalanced training set.


In [ ]:
# ==========================================================
# Solution 2.1: Build CNN [5 pts]
# ==========================================================
def build_cnn(img_size=IMG_SIZE):
    model = keras.Sequential([
        layers.Input(shape=(img_size, img_size, 1)),
        layers.Conv2D(32,3,padding="same",activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(64,3,padding="same",activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(128,3,padding="same",activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(256,3,padding="same",activation="relu"),
        layers.BatchNormalization(), layers.GlobalAveragePooling2D(),
        layers.Dense(128,activation="relu"), layers.Dropout(0.4),
        layers.Dense(1,activation="sigmoid"),
    ], name="pneumonia_cnn")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
# ==========================================================
# Solution 2.2: Load pretrained baseline checkpoint and fine-tune [10 pts]
# ==========================================================
reset_all_seeds(SEED + 1000)
validate_checkpoint(BASELINE_PRETRAINED_PATH)
baseline_model = keras.saving.load_model(BASELINE_PRETRAINED_PATH)
baseline_model = compile_classifier_for_key_finetune(baseline_model)

train_ds = make_tf_dataset(imb_train_images, imb_train_labels)
val_ds = make_tf_dataset(val_images, val_labels, shuffle=False)

baseline_history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=KEY_FINE_TUNE_EPOCHS,
    verbose=1,
)


In [ ]:
# ==========================================================
# Solution 2.3: Evaluate and visualize predictions [5 pts]
# ==========================================================
baseline_metrics = evaluate_model(baseline_model, test_images, test_labels, "Baseline CNN")
plot_confusion_matrix(np.array(baseline_metrics["confusion_matrix"]), "Baseline CNN")
plot_prediction_examples(baseline_model, test_images, test_labels, n=8, title="Baseline CNN — Test Predictions")

## Part 3 — Traditional Augmentation CNN with Checkpoint Loading and Fine-Tuning [15 pts]

Traditional augmentation is applied only during training. Validation and test images remain unchanged.


In [ ]:
# ==========================================================
# Solution 3.1: Augmentation pipeline [5 pts]
# ==========================================================
aug_pipeline = build_augmentation_pipeline()

print("Traditional augmentation config:")
print(f"  rotation factor:    {TRAD_ROTATION_FACTOR}")
print(f"  translation factor: {TRAD_TRANSLATION_FACTOR}")
print(f"  zoom factor:        {TRAD_ZOOM_FACTOR}")
print(f"  contrast factor:    {TRAD_CONTRAST_FACTOR}")
print(f"  fill mode:          {TRAD_FILL_MODE}")


Traditional augmentation uses small label-preserving changes such as slight rotation, translation, zoom, and contrast variation. These approximate realistic acquisition/positioning variation without changing the class label.

In [ ]:
# ==========================================================
# Solution 3.2: Visualize traditional augmentation examples
# ==========================================================
minority_mask_vis = imb_train_labels == MINORITY_CLASS
minority_vis = imb_train_images[minority_mask_vis][:4]
plot_traditional_aug_examples(minority_vis, aug_pipeline, n=4, repeats=3)

In [ ]:
# ==========================================================
# Solution 3.3: Load pretrained traditional-aug checkpoint and fine-tune [5 pts]
# ==========================================================
reset_all_seeds(SEED + 2000)
validate_checkpoint(TRAD_AUG_PRETRAINED_PATH)
trad_aug_model = keras.saving.load_model(TRAD_AUG_PRETRAINED_PATH)
trad_aug_model = compile_classifier_for_key_finetune(trad_aug_model)

aug_train_ds = make_tf_dataset(imb_train_images, imb_train_labels, augment_fn=aug_pipeline)
trad_aug_history = trad_aug_model.fit(
    aug_train_ds,
    validation_data=val_ds,
    epochs=KEY_FINE_TUNE_EPOCHS,
    verbose=1,
)


In [ ]:
# ==========================================================
# Solution 3.4: Evaluate [5 pts]
# ==========================================================
trad_aug_metrics = evaluate_model(trad_aug_model, test_images, test_labels, "Traditional Aug CNN")
plot_confusion_matrix(np.array(trad_aug_metrics["confusion_matrix"]), "Traditional Aug CNN")

## Part 4 — Diffusion-Style Masked Denoising/Inpainting Augmentation [25 pts]

The denoiser checkpoint is loaded from the provided `denoiser_final.keras` file. The denoiser is fine-tuned briefly on the imbalanced training-set pneumonia samples, then used to generate additional minority-class training images.

Generated images are added only to the training set. Validation and test sets remain unchanged.


### Diffusion Helpers (Provided)

In [ ]:
# ── Diffusion schedule ──
betas_np = np.linspace(NOISE_BETA_START, NOISE_BETA_END, DIFFUSION_TIMESTEPS, dtype=np.float32)
alphas_np = 1.0 - betas_np
alpha_bars_np = np.cumprod(alphas_np).astype(np.float32)
BETAS = tf.constant(betas_np)
ALPHAS = tf.constant(alphas_np)
ALPHA_BARS = tf.constant(alpha_bars_np)


def q_sample(x0, t, noise=None):
    if noise is None:
        noise = tf.random.normal(tf.shape(x0))
    ab = tf.gather(ALPHA_BARS, t)
    while len(ab.shape) < len(x0.shape):
        ab = ab[..., tf.newaxis]
    return tf.sqrt(ab) * x0 + tf.sqrt(1.0 - ab) * noise


@keras.saving.register_keras_serializable(package="DiffusionInpainting")
class SinusoidalTimeEmbedding(layers.Layer):
    def __init__(self, dim=64, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim

    def call(self, t):
        half = self.dim // 2
        freqs = tf.exp(-tf.math.log(10000.0) * tf.range(half, dtype=tf.float32) / half)
        t_f = tf.cast(tf.reshape(t, [-1]), tf.float32)
        args = t_f[:, tf.newaxis] * freqs[tf.newaxis, :]
        return tf.concat([tf.sin(args), tf.cos(args)], axis=-1)

    def get_config(self):
        config = super().get_config()
        config.update({"dim": self.dim})
        return config


def reverse_step(model, x_t, t):
    n = tf.shape(x_t)[0]
    t_batch = tf.fill([n], t)
    pred_noise = model([x_t, t_batch], training=False)
    alpha = ALPHAS[t]
    alpha_bar = ALPHA_BARS[t]
    beta = BETAS[t]
    mean = (1.0 / tf.sqrt(alpha)) * (x_t - (beta / tf.sqrt(1.0 - alpha_bar)) * pred_noise)
    if t > 0:
        return mean + tf.sqrt(beta) * tf.random.normal(tf.shape(x_t))
    return mean


def generate_inpainted_samples(model, source_images, n_per_image=1, mask_frac=INPAINT_MASK_FRAC):
    results = []
    for src in source_images:
        src = tf.cast(src, tf.float32)
        if len(src.shape) == 2:
            src = src[..., tf.newaxis]
        for _ in range(n_per_image):
            h = w = IMG_SIZE
            mh = int(h * mask_frac)
            mw = int(w * mask_frac)
            top = np.random.randint(0, h - mh + 1)
            left = np.random.randint(0, w - mw + 1)
            mask = np.zeros((h, w, 1), dtype=np.float32)
            mask[top:top + mh, left:left + mw, :] = 1.0
            mask_tf = tf.constant(mask)
            x = tf.random.normal((1, IMG_SIZE, IMG_SIZE, 1))
            src_b = src[tf.newaxis]
            for t in reversed(range(DIFFUSION_TIMESTEPS)):
                x = reverse_step(model, x, t)
                if t > 0:
                    noised_src = q_sample(src_b, tf.constant([t]))
                    x = mask_tf * x + (1.0 - mask_tf) * noised_src
                else:
                    x = mask_tf * x + (1.0 - mask_tf) * src_b
            results.append(tf.clip_by_value(x[0], 0.0, 1.0))
    return tf.stack(results)


def generate_inpainted_with_fixed_mask(model, source_image, mask):
    """Generate one inpainted image using a provided binary mask.
    mask = 1 inside the region to generate; mask = 0 outside the region to preserve.
    """
    src = tf.cast(source_image, tf.float32)
    if len(src.shape) == 2:
        src = src[..., tf.newaxis]
    src_b = src[tf.newaxis]
    mask_tf = tf.constant(mask.astype(np.float32))
    x = tf.random.normal((1, IMG_SIZE, IMG_SIZE, 1))
    for t in reversed(range(DIFFUSION_TIMESTEPS)):
        x = reverse_step(model, x, t)
        if t > 0:
            anchor_noise = tf.random.normal(tf.shape(src_b))
            noised_src = q_sample(src_b, tf.constant([t]), anchor_noise)
            x = mask_tf * x + (1.0 - mask_tf) * noised_src
        else:
            x = mask_tf * x + (1.0 - mask_tf) * src_b
    return tf.clip_by_value(x[0], 0.0, 1.0)


def make_soft_mask(mask, blur_size=5):
    """Create a softly blended version of a binary mask for visualization only."""
    mask_tf = tf.convert_to_tensor(mask[tf.newaxis], dtype=tf.float32)
    for _ in range(2):
        mask_tf = tf.nn.avg_pool2d(mask_tf, ksize=blur_size, strides=1, padding="SAME")
    return tf.clip_by_value(mask_tf[0], 0.0, 1.0).numpy()


def plot_diffusion_inpainting_examples(denoiser, source_images, n=4, mask_frac=INPAINT_MASK_FRAC):
    """Visualize original → same mask → inpainted output.
    A soft mask is used only for display so the generated patch blends more smoothly.
    """
    n = min(n, len(source_images))
    fig, axes = plt.subplots(n, 3, figsize=(6, n * 2))
    if n == 1:
        axes = axes[np.newaxis, :]
    for i in range(n):
        src = source_images[i]
        src_2d = np.squeeze(src)
        h = w = IMG_SIZE
        mh = int(h * mask_frac)
        mw = int(w * mask_frac)
        top = np.random.randint(0, h - mh + 1)
        left = np.random.randint(0, w - mw + 1)
        mask = np.zeros((h, w, 1), dtype=np.float32)
        mask[top:top + mh, left:left + mw, :] = 1.0
        masked_vis = src_2d.copy()
        masked_vis[top:top + mh, left:left + mw] = 0.5

        raw_inpainted = generate_inpainted_with_fixed_mask(denoiser, src, mask)
        soft_mask = make_soft_mask(mask)
        display_inpainted = soft_mask * raw_inpainted.numpy() + (1.0 - soft_mask) * src
        display_inpainted = np.clip(display_inpainted, 0.0, 1.0)

        axes[i, 0].imshow(src_2d, cmap="gray")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(masked_vis, cmap="gray")
        axes[i, 1].axis("off")
        axes[i, 2].imshow(np.squeeze(display_inpainted), cmap="gray")
        axes[i, 2].axis("off")
    axes[0, 0].set_title("Original", fontsize=9)
    axes[0, 1].set_title("Same Mask Used\nfor Inpainting", fontsize=9)
    axes[0, 2].set_title("Inpainted Output", fontsize=9)
    fig.suptitle("Diffusion Inpainting: Original → Same Mask → Inpainted")
    plt.tight_layout()
    plt.show()


print("Diffusion helpers loaded.")


In [ ]:
# ==========================================================
# Solution 4.1: Load pretrained denoiser and fine-tune briefly [8 pts]
# ==========================================================
reset_all_seeds(SEED + 2500)
validate_checkpoint(DENOISER_PRETRAINED_PATH, is_denoiser=True)
denoiser = keras.saving.load_model(DENOISER_PRETRAINED_PATH)
denoiser.compile(optimizer=keras.optimizers.Adam(KEY_DENOISER_LR), loss="mse")

minority_mask = imb_train_labels == MINORITY_CLASS
minority_images = imb_train_images[minority_mask]
print(f"Fine-tuning denoiser on {len(minority_images)} minority samples")

for ep in range(KEY_FINE_TUNE_EPOCHS):
    losses = []
    idx = np.arange(len(minority_images))
    np.random.shuffle(idx)
    for s in range(0, len(minority_images), BATCH_SIZE):
        batch = tf.constant(minority_images[idx[s:s + BATCH_SIZE]])
        t = tf.random.uniform([len(batch)], 0, DIFFUSION_TIMESTEPS, dtype=tf.int32)
        noise = tf.random.normal(tf.shape(batch))
        noisy = q_sample(batch, t, noise)
        loss = denoiser.train_on_batch([noisy, t], noise)
        losses.append(float(np.asarray(loss).item()))
    print(f"  Epoch {ep + 1}/{KEY_FINE_TUNE_EPOCHS} — loss: {np.mean(losses):.6f}")


The same mask is used for the masked panel and the inpainted output, so the visualization accurately shows what region was generated.

In [ ]:
# ==========================================================
# Solution 4.2: Visualize the inpainting process
# ==========================================================
plot_original_samples(minority_images, np.full(len(minority_images[:8]), MINORITY_CLASS, dtype=np.int32),
                      n=8, title="Original Minority Samples")
plot_diffusion_inpainting_examples(denoiser, minority_images[:4], n=4, mask_frac=INPAINT_MASK_FRAC)

In [ ]:
# ==========================================================
# Solution 4.3: Generate synthetic minority samples [7 pts]
# ==========================================================
n_minority_current = int(np.sum(imb_train_labels == MINORITY_CLASS))
n_target = n_minority_current * TARGET_MINORITY_MULTIPLIER
n_to_generate = max(0, n_target - n_minority_current)
print(f"Current minority: {n_minority_current}, target: {n_target}, generating: {n_to_generate}")

if n_to_generate > 0:
    seed_idx = np.random.choice(len(minority_images), n_to_generate, replace=True)
    seed_imgs = minority_images[seed_idx]
    generated_images = generate_inpainted_samples(
        denoiser,
        seed_imgs,
        n_per_image=1,
        mask_frac=INPAINT_MASK_FRAC,
    ).numpy()
    generated_labels = np.full(len(generated_images), MINORITY_CLASS, dtype=np.int32)
else:
    generated_images = np.empty((0, IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
    generated_labels = np.array([], dtype=np.int32)

print(f"Generated {len(generated_images)} samples, shape: {generated_images.shape}")
plot_image_grid(generated_images, n=8, title="Generated Minority Samples (Inpainted)")


In [ ]:
# ==========================================================
# Solution 4.4: Load pretrained diffusion-augmented CNN and fine-tune [10 pts]
# ==========================================================
diff_train_images = np.concatenate([imb_train_images, generated_images], axis=0).astype(np.float32)
diff_train_labels = np.concatenate([imb_train_labels, generated_labels], axis=0).astype(np.int32)
perm = np.random.permutation(len(diff_train_images))
diff_train_images, diff_train_labels = diff_train_images[perm], diff_train_labels[perm]
print(f"Diffusion-augmented train set: {len(diff_train_images)} samples")
print("Class counts:", np.bincount(diff_train_labels, minlength=2))

reset_all_seeds(SEED + 3000)
validate_checkpoint(DIFFUSION_AUG_PRETRAINED_PATH)
diff_aug_model = keras.saving.load_model(DIFFUSION_AUG_PRETRAINED_PATH)
diff_aug_model = compile_classifier_for_key_finetune(diff_aug_model)

diff_train_ds = make_tf_dataset(diff_train_images, diff_train_labels)
diff_aug_history = diff_aug_model.fit(
    diff_train_ds,
    validation_data=val_ds,
    epochs=KEY_FINE_TUNE_EPOCHS,
    verbose=1,
)

diff_aug_metrics = evaluate_model(diff_aug_model, test_images, test_labels, "Diffusion Aug CNN")
plot_confusion_matrix(np.array(diff_aug_metrics["confusion_matrix"]), "Diffusion Aug CNN")


## Part 5 — Metrics and Model Comparison [20 pts]

The table below reports metrics computed in this Key notebook run after brief fine-tuning. These are the values that should be discussed for this run.


In [ ]:
# ==========================================================
# Solution 5.1: Comparison table [10 pts]
# ==========================================================
all_metrics = {
    "Baseline CNN": baseline_metrics,
    "Traditional Aug CNN": trad_aug_metrics,
    "Diffusion Aug CNN": diff_aug_metrics,
}

print("Metrics from this Key notebook run after fine-tuning:")
print(f"{'Model':<25} {'Acc':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")
print("-" * 57)
for nm, m in all_metrics.items():
    print(
        f"{nm:<25} {m['accuracy']:>8.4f} {m['macro_f1']:>8.4f} "
        f"{m['minority_precision']:>8.4f} {m['minority_recall']:>8.4f}"
    )


In [ ]:
# ==========================================================
# Solution 5.2: Side-by-side confusion matrices [5 pts]
# ==========================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nm, m) in zip(axes, all_metrics.items()):
    cm = np.array(m["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["normal","pneumonia"]); ax.set_yticklabels(["normal","pneumonia"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(nm)
    for i in range(2):
        for j in range(2):
            ax.text(j,i,str(cm[i][j]),ha="center",va="center",
                    color="white" if cm[i][j]>cm.max()/2 else "black", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ==========================================================
# Solution 5.3: Classification reports [3 pts]
# ==========================================================
for nm, mdl in [("Baseline", baseline_model), ("Traditional Aug", trad_aug_model),
                ("Diffusion Aug", diff_aug_model)]:
    preds = (mdl.predict(test_images, verbose=0).squeeze() >= 0.5).astype(int)
    print(f"\n{'='*50}\n  {nm} — Classification Report\n{'='*50}")
    print(classification_report(test_labels, preds, target_names=["normal","pneumonia"]))

In [ ]:
# ==========================================================
# Solution 5.4: Final augmentation strategy comparison [2 pts]
# ==========================================================
print(
    "Note: the traditional column is a transformed version of the same original image. "
    "The diffusion column shows generated minority samples from the diffusion pool."
)
plot_augmentation_strategy_comparison(minority_images, aug_pipeline, generated_images, n=6)


## Part 6 — Reflection [5 pts]

### Model Reflection Answers

## Grading Rubric Summary (100 pts)

| Part | Section                                               | Points |
|----:|--------------------------------------------------------|------:|
| 1 | Load PneumoniaMNIST and create artificial imbalance      | 15 |
| 2 | Baseline CNN with checkpoint loading and fine-tuning      | 20 |
| 3 | Traditional augmentation CNN with checkpoint loading      | 15 |
| 4 | Diffusion inpainting augmentation workflow               | 25 |
| 5 | Metrics and model comparison                             | 20 |
| 6 | Reflection                                               | 5 |
|   | **Total**                                                | **100** |
